In [1]:
# Print scores
def print_scores(scores):
    for score in sorted(scores.keys()):
        fields = scores[score]
        print("{}: {}".format(score, ", ".join(fields)))

# Add score
def add_score(scores, score, fields):
    if score not in scores:
        scores[score] = []
    
    for field in fields:
        scores[score].append(field)

# Add scores
def add_scores(old_scores, new_scores):
    for score in new_scores:
        add_score(old_scores, score, new_scores[score])

In [2]:
# Generate basic scores
def primitive_scores(prefix, factor):
    scores = {}
    for number in range(1, 21):
        field = prefix + str(number)
        score = factor * number
        add_score(scores, score, [field])
    return scores

# Initialize basic score prefixes
prefixes = ["", "D", "T"]

# Initialize scores variables
single_scores = {}
double_scores = {}
triple_scores = {}

In [3]:
# Store basic scores
single_scores = primitive_scores(prefixes[0], 1)
double_scores = primitive_scores(prefixes[1], 2)
triple_scores = primitive_scores(prefixes[2], 3)

# Add special scores
add_score(single_scores, 25, ["25"])
add_score(double_scores, 50, ["BULL"])

# Show basic and special scores
print("Single scores:")
print_scores(single_scores)
print("Double scores:")
print_scores(double_scores)
print("Triple scores:")
print_scores(triple_scores)

Single scores:
1: 1
2: 2
3: 3
4: 4
5: 5
6: 6
7: 7
8: 8
9: 9
10: 10
11: 11
12: 12
13: 13
14: 14
15: 15
16: 16
17: 17
18: 18
19: 19
20: 20
25: 25
Double scores:
2: D1
4: D2
6: D3
8: D4
10: D5
12: D6
14: D7
16: D8
18: D9
20: D10
22: D11
24: D12
26: D13
28: D14
30: D15
32: D16
34: D17
36: D18
38: D19
40: D20
50: BULL
Triple scores:
3: T1
6: T2
9: T3
12: T4
15: T5
18: T6
21: T7
24: T8
27: T9
30: T10
33: T11
36: T12
39: T13
42: T14
45: T15
48: T16
51: T17
54: T18
57: T19
60: T20


In [4]:
# Initialize aggregate scores
master_scores ={}
add_scores(master_scores, double_scores)
add_scores(master_scores, triple_scores)

all_scores = {}
add_scores(all_scores, single_scores)
add_scores(all_scores, double_scores)
add_scores(all_scores, triple_scores)

# Show aggregate scores
print("Master scores:")
print_scores(master_scores)
print("All scores:")
print_scores(all_scores)

Master scores:
2: D1
3: T1
4: D2
6: D3, T2
8: D4
9: T3
10: D5
12: D6, T4
14: D7
15: T5
16: D8
18: D9, T6
20: D10
21: T7
22: D11
24: D12, T8
26: D13
27: T9
28: D14
30: D15, T10
32: D16
33: T11
34: D17
36: D18, T12
38: D19
39: T13
40: D20
42: T14
45: T15
48: T16
50: BULL
51: T17
54: T18
57: T19
60: T20
All scores:
1: 1
2: 2, D1
3: 3, T1
4: 4, D2
5: 5
6: 6, D3, T2
7: 7
8: 8, D4
9: 9, T3
10: 10, D5
11: 11
12: 12, D6, T4
13: 13
14: 14, D7
15: 15, T5
16: 16, D8
17: 17
18: 18, D9, T6
19: 19
20: 20, D10
21: T7
22: D11
24: D12, T8
25: 25
26: D13
27: T9
28: D14
30: D15, T10
32: D16
33: T11
34: D17
36: D18, T12
38: D19
39: T13
40: D20
42: T14
45: T15
48: T16
50: BULL
51: T17
54: T18
57: T19
60: T20


In [5]:
# Set darts scores
first_scores = all_scores
middle_scores = all_scores
last_scores = double_scores

# Set number of darts
first_darts = 0
middle_darts = 1
last_darts = 0

In [6]:
# Path overview
def print_summary(steps, goal, paths):
    global rec_calls, failed_paths

    num_base_steps = [len(base_steps) for base_steps in steps]
    total_paths = multiply(num_base_steps)
    found_paths = len(paths)
    skipped_paths = total_paths - found_paths - failed_paths

    print(
        f"Goal: {goal}\n"
        f"Max recursive calls: {max_rec_calls(steps)}\n"
        f"Num recursive calls: {rec_calls}\n"
        f"Total paths: {total_paths}\n"
        f"Found paths: {found_paths}\n"
        f"Failed paths: {failed_paths}\n"
        f"Failed percentage: {failed_paths / total_paths * 100}\n"
        f"Skipped paths: {skipped_paths}\n"
        f"Skipped percentage: {skipped_paths / total_paths * 100}\n"
        f"Paths: {paths}\n"
    )

def print_stats_summary(steps, goal, paths, stats):
    num_base_steps = [len(base_steps) for base_steps in steps]
    total_paths = multiply(num_base_steps)
    found_paths = len(paths)
    failed_paths = stats["fails"]
    skipped_paths = total_paths - found_paths - failed_paths

    print(
        f"Goal: {goal}\n"
        f"Max recursive calls: {max_rec_calls(steps)}\n"
        f"Num recursive calls: {stats['calls']}\n"
        f"Num ifs: {stats['ifs']}\n"
        f"Num pre overshoots: {stats['preovershoots']}\n"
        f"Num pre undershoots: {stats['preundershoots']}\n"
        f"Num for steps: {stats['fors']}\n"
        f"Num for over bound: {stats['foroverbound']}\n"
        f"Num for overshoots: {stats['forovershoots']}\n"
        f"Num for undershoots: {stats['forundershoots']}\n"
        f"Total paths: {total_paths}\n"
        f"Found paths: {found_paths}\n"
        f"Failed paths: {failed_paths}\n"
        f"Failed percentage: {failed_paths / total_paths * 100}\n"
        f"Skipped paths: {skipped_paths}\n"
        f"Skipped percentage: {skipped_paths / total_paths * 100}\n"
        f"Paths: {paths}\n"
    )

In [7]:
# Find all paths
def find_all_paths(steps):
    step_num = 0
    path = []
    paths = []
    find_all_paths_rec(steps, step_num, path, paths)
    return paths

def find_all_paths_rec(steps, step_num, path, paths):
    global rec_calls
    rec_calls += 1

    # print(f"Path: {path}")

    if step_num == len(steps):
        paths.append(list(path))
        return
    else:
        for step in steps[step_num]:
            path.append(step)
            find_all_paths_rec(steps, step_num + 1, path, paths)
            path.pop()

def multiply(nums):
    if len(nums) == 0:
        return 0
    else:
        prod = 1
        for num in nums:
            prod *= num

        return prod

def max_rec_calls(steps):
    num_base_steps = [len(base_steps) for base_steps in steps]

    rec_calls = 1

    for end in range(1, len(num_base_steps) + 1):
        rec_calls += multiply(num_base_steps[:end])

    return rec_calls

In [8]:
# Find all paths (test)
rec_calls = 0
failed_paths = 0
base_steps = [1, 2, 3]
steps = [base_steps, ]*3
total_paths = len(base_steps) ** len(steps)
paths = find_all_paths(steps)
print_summary(steps, "X", paths)

Goal: X
Max recursive calls: 40
Num recursive calls: 40
Total paths: 27
Found paths: 27
Failed paths: 0
Failed percentage: 0.0
Skipped paths: 0
Skipped percentage: 0.0
Paths: [[1, 1, 1], [1, 1, 2], [1, 1, 3], [1, 2, 1], [1, 2, 2], [1, 2, 3], [1, 3, 1], [1, 3, 2], [1, 3, 3], [2, 1, 1], [2, 1, 2], [2, 1, 3], [2, 2, 1], [2, 2, 2], [2, 2, 3], [2, 3, 1], [2, 3, 2], [2, 3, 3], [3, 1, 1], [3, 1, 2], [3, 1, 3], [3, 2, 1], [3, 2, 2], [3, 2, 3], [3, 3, 1], [3, 3, 2], [3, 3, 3]]



In [9]:
# Find goal paths
def find_goal_paths(steps, goal):
    step_num = 0
    path = []
    paths = []
    find_goal_paths_rec(steps, goal, step_num, path, paths)
    return paths

def find_goal_paths_rec(steps, goal, step_num, path, paths):
    global rec_calls, failed_paths
    rec_calls += 1

    if step_num == len(steps):
        if sum(path) == goal:
            paths.append(list(path))
        else:
            failed_paths += 1
        return
    else:
        for step in steps[step_num]:
            path.append(step)
            find_goal_paths_rec(steps, goal, step_num + 1, path, paths)
            path.pop()

In [10]:
# Find goal paths (test)
for goal in range(2, 8):
    rec_calls = 0
    failed_paths = 0
    base_steps = [1, 2]
    steps = [base_steps, ]*3
    total_paths = len(base_steps) ** len(steps)
    paths = find_goal_paths(steps, goal)

    print_summary(steps, goal, paths)

Goal: 2
Max recursive calls: 15
Num recursive calls: 15
Total paths: 8
Found paths: 0
Failed paths: 8
Failed percentage: 100.0
Skipped paths: 0
Skipped percentage: 0.0
Paths: []

Goal: 3
Max recursive calls: 15
Num recursive calls: 15
Total paths: 8
Found paths: 1
Failed paths: 7
Failed percentage: 87.5
Skipped paths: 0
Skipped percentage: 0.0
Paths: [[1, 1, 1]]

Goal: 4
Max recursive calls: 15
Num recursive calls: 15
Total paths: 8
Found paths: 3
Failed paths: 5
Failed percentage: 62.5
Skipped paths: 0
Skipped percentage: 0.0
Paths: [[1, 1, 2], [1, 2, 1], [2, 1, 1]]

Goal: 5
Max recursive calls: 15
Num recursive calls: 15
Total paths: 8
Found paths: 3
Failed paths: 5
Failed percentage: 62.5
Skipped paths: 0
Skipped percentage: 0.0
Paths: [[1, 2, 2], [2, 1, 2], [2, 2, 1]]

Goal: 6
Max recursive calls: 15
Num recursive calls: 15
Total paths: 8
Found paths: 1
Failed paths: 7
Failed percentage: 87.5
Skipped paths: 0
Skipped percentage: 0.0
Paths: [[2, 2, 2]]

Goal: 7
Max recursive calls: 

In [11]:
# Find goal paths (three darts, 180)
rec_calls = 0
failed_paths = 0
base_steps = all_scores.keys()
steps = [base_steps, ]*3
total_paths = len(base_steps) ** len(steps)
paths = find_goal_paths(steps, 180)

print_summary(steps, goal, paths)

Goal: 7
Max recursive calls: 81400
Num recursive calls: 81400
Total paths: 79507
Found paths: 1
Failed paths: 79506
Failed percentage: 99.99874224911014
Skipped paths: 0
Skipped percentage: 0.0
Paths: [[60, 60, 60]]



In [12]:
# Find goal paths (three darts, 170 - 180)
for goal in range(180, 169, -1):
    rec_calls = 0
    failed_paths = 0
    base_steps = all_scores.keys()
    steps = [base_steps, ]*3
    total_paths = len(base_steps) ** len(steps)
    paths = find_goal_paths(steps, goal)

    print_summary(steps, goal, paths)

Goal: 180
Max recursive calls: 81400
Num recursive calls: 81400
Total paths: 79507
Found paths: 1
Failed paths: 79506
Failed percentage: 99.99874224911014
Skipped paths: 0
Skipped percentage: 0.0
Paths: [[60, 60, 60]]

Goal: 179
Max recursive calls: 81400
Num recursive calls: 81400
Total paths: 79507
Found paths: 0
Failed paths: 79507
Failed percentage: 100.0
Skipped paths: 0
Skipped percentage: 0.0
Paths: []

Goal: 178
Max recursive calls: 81400
Num recursive calls: 81400
Total paths: 79507
Found paths: 0
Failed paths: 79507
Failed percentage: 100.0
Skipped paths: 0
Skipped percentage: 0.0
Paths: []

Goal: 177
Max recursive calls: 81400
Num recursive calls: 81400
Total paths: 79507
Found paths: 3
Failed paths: 79504
Failed percentage: 99.99622674733043
Skipped paths: 0
Skipped percentage: 0.0
Paths: [[57, 60, 60], [60, 57, 60], [60, 60, 57]]

Goal: 176
Max recursive calls: 81400
Num recursive calls: 81400
Total paths: 79507
Found paths: 0
Failed paths: 79507
Failed percentage: 100.0
S

In [13]:
# Find bounded goal paths
def find_bounded_goal_paths(steps, goal):
    # Sort steps in decreasing order
    steps = [sorted(base_steps, reverse=True) for base_steps in steps]

    # Initialize recursion
    step_num = 0
    step_bound = 1000
    path = []
    paths = []
    find_bounded_goal_paths_rec(steps, goal, step_num, step_bound, path, paths)

    return paths

def find_bounded_goal_paths_rec(steps, goal, step_num, step_bound, path, paths):
    global rec_calls, failed_paths
    rec_calls += 1

    if step_num == len(steps):
        if sum(path) == goal:
            paths.append(list(path))
        else:
            failed_paths += 1
        return
    else:
        for step in steps[step_num]:
            if step > step_bound:
                continue

            path.append(step)
            find_bounded_goal_paths_rec(steps, goal, step_num + 1, step, path, paths)
            path.pop()

In [14]:
# Find bounded goal paths (test)
for goal in range(2, 8):
    rec_calls = 0
    failed_paths = 0
    base_steps = [1, 2]
    steps = [base_steps, ]*3
    total_paths = len(base_steps) ** len(steps)
    paths = find_bounded_goal_paths(steps, goal)

    print_summary(steps, goal, paths)

Goal: 2
Max recursive calls: 15
Num recursive calls: 10
Total paths: 8
Found paths: 0
Failed paths: 4
Failed percentage: 50.0
Skipped paths: 4
Skipped percentage: 50.0
Paths: []

Goal: 3
Max recursive calls: 15
Num recursive calls: 10
Total paths: 8
Found paths: 1
Failed paths: 3
Failed percentage: 37.5
Skipped paths: 4
Skipped percentage: 50.0
Paths: [[1, 1, 1]]

Goal: 4
Max recursive calls: 15
Num recursive calls: 10
Total paths: 8
Found paths: 1
Failed paths: 3
Failed percentage: 37.5
Skipped paths: 4
Skipped percentage: 50.0
Paths: [[2, 1, 1]]

Goal: 5
Max recursive calls: 15
Num recursive calls: 10
Total paths: 8
Found paths: 1
Failed paths: 3
Failed percentage: 37.5
Skipped paths: 4
Skipped percentage: 50.0
Paths: [[2, 2, 1]]

Goal: 6
Max recursive calls: 15
Num recursive calls: 10
Total paths: 8
Found paths: 1
Failed paths: 3
Failed percentage: 37.5
Skipped paths: 4
Skipped percentage: 50.0
Paths: [[2, 2, 2]]

Goal: 7
Max recursive calls: 15
Num recursive calls: 10
Total paths: 

In [15]:
# Find bounded goal paths (three darts, 170 - 180)
darts = 3
goals = range(180, 169, -1)
for goal in goals:
    rec_calls = 0
    failed_paths = 0
    base_steps = all_scores.keys()
    steps = [base_steps, ]*darts
    paths = find_bounded_goal_paths(steps, goal)

    print_summary(steps, goal, paths)

Goal: 180
Max recursive calls: 81400
Num recursive calls: 15180
Total paths: 79507
Found paths: 1
Failed paths: 14189
Failed percentage: 17.84622737620587
Skipped paths: 65317
Skipped percentage: 82.15251487290428
Paths: [[60, 60, 60]]

Goal: 179
Max recursive calls: 81400
Num recursive calls: 15180
Total paths: 79507
Found paths: 0
Failed paths: 14190
Failed percentage: 17.847485127095727
Skipped paths: 65317
Skipped percentage: 82.15251487290428
Paths: []

Goal: 178
Max recursive calls: 81400
Num recursive calls: 15180
Total paths: 79507
Found paths: 0
Failed paths: 14190
Failed percentage: 17.847485127095727
Skipped paths: 65317
Skipped percentage: 82.15251487290428
Paths: []

Goal: 177
Max recursive calls: 81400
Num recursive calls: 15180
Total paths: 79507
Found paths: 1
Failed paths: 14189
Failed percentage: 17.84622737620587
Skipped paths: 65317
Skipped percentage: 82.15251487290428
Paths: [[60, 60, 57]]

Goal: 176
Max recursive calls: 81400
Num recursive calls: 15180
Total path

In [16]:
# Find bounded goal paths (five darts, 300)
goals = [300]
darts = 5
for goal in goals:
    rec_calls = 0
    failed_paths = 0
    base_steps = all_scores.keys()
    steps = [base_steps, ]*5
    paths = find_bounded_goal_paths(steps, goal)

    print_summary(steps, goal, paths)

Goal: 300
Max recursive calls: 150508644
Num recursive calls: 1712304
Total paths: 147008443
Found paths: 1
Failed paths: 1533938
Failed percentage: 1.0434353079979224
Skipped paths: 145474504
Skipped percentage: 98.95656401176905
Paths: [[60, 60, 60, 60, 60]]



In [17]:
# Find bounded goal paths (six darts, 301)
goals = [301]
darts = 6
for goal in goals:
    rec_calls = 0
    failed_paths = 0
    base_steps = all_scores.keys()
    steps = [base_steps, ]*darts
    paths = find_bounded_goal_paths(steps, goal)
    print_summary(steps, goal, paths)

Goal: 301
Max recursive calls: 6471871693
Num recursive calls: 13983816
Total paths: 6321363049
Found paths: 277
Failed paths: 12271235
Failed percentage: 0.19412324375106463
Skipped paths: 6309091537
Skipped percentage: 99.80587237428261
Paths: [[60, 60, 60, 60, 60, 1], [60, 60, 60, 60, 57, 4], [60, 60, 60, 60, 54, 7], [60, 60, 60, 60, 51, 10], [60, 60, 60, 60, 50, 11], [60, 60, 60, 60, 48, 13], [60, 60, 60, 60, 45, 16], [60, 60, 60, 60, 42, 19], [60, 60, 60, 60, 40, 21], [60, 60, 60, 60, 39, 22], [60, 60, 60, 60, 36, 25], [60, 60, 60, 60, 34, 27], [60, 60, 60, 60, 33, 28], [60, 60, 60, 57, 57, 7], [60, 60, 60, 57, 54, 10], [60, 60, 60, 57, 51, 13], [60, 60, 60, 57, 50, 14], [60, 60, 60, 57, 48, 16], [60, 60, 60, 57, 45, 19], [60, 60, 60, 57, 42, 22], [60, 60, 60, 57, 40, 24], [60, 60, 60, 57, 39, 25], [60, 60, 60, 57, 38, 26], [60, 60, 60, 57, 36, 28], [60, 60, 60, 57, 34, 30], [60, 60, 60, 57, 32, 32], [60, 60, 60, 54, 54, 13], [60, 60, 60, 54, 51, 16], [60, 60, 60, 54, 50, 17], [60

In [18]:
# Find bounded goal paths (one dart, 60)
darts = 1
goals = [60]
for goal in goals:
    rec_calls = 0
    failed_paths = 0
    base_steps = all_scores.keys()
    steps = [base_steps, ]*darts
    paths = find_bounded_goal_paths(steps, goal)
    print_summary(steps, goal, paths)

Goal: 60
Max recursive calls: 44
Num recursive calls: 44
Total paths: 43
Found paths: 1
Failed paths: 42
Failed percentage: 97.67441860465115
Skipped paths: 0
Skipped percentage: 0.0
Paths: [[60]]



In [19]:
# Find bounded goal paths (two darts, 60)
darts = 2
goals = [60]
for goal in goals:
    rec_calls = 0
    failed_paths = 0
    base_steps = all_scores.keys()
    steps = [base_steps, ]*darts
    paths = find_bounded_goal_paths(steps, goal)
    print(f"Goal: {goal:1} Num recursive calls: {rec_calls} Max num recursive calls: {max_rec_calls(steps)} Num paths: {len(paths):1} Paths: {paths}\n")
    print_summary(steps, goal, paths)

Goal: 60 Num recursive calls: 990 Max num recursive calls: 1893 Num paths: 15 Paths: [[57, 3], [54, 6], [51, 9], [50, 10], [48, 12], [45, 15], [42, 18], [40, 20], [39, 21], [38, 22], [36, 24], [34, 26], [33, 27], [32, 28], [30, 30]]

Goal: 60
Max recursive calls: 1893
Num recursive calls: 990
Total paths: 1849
Found paths: 15
Failed paths: 931
Failed percentage: 50.35154137371553
Skipped paths: 903
Skipped percentage: 48.837209302325576
Paths: [[57, 3], [54, 6], [51, 9], [50, 10], [48, 12], [45, 15], [42, 18], [40, 20], [39, 21], [38, 22], [36, 24], [34, 26], [33, 27], [32, 28], [30, 30]]



In [20]:
# Find bounded targeted goal paths
def max_remaining(steps, step_num):
    selected_steps = steps[step_num:]
    return sum([max(base_steps) for base_steps in selected_steps])

def min_remaining(steps, step_num):
    selected_steps = steps[step_num:]
    return sum([min(base_steps) for base_steps in selected_steps])

def find_bounded_targeted_goal_paths(steps, goal):
    # Preprocess steps
    steps = [sorted(base_steps, reverse=True) for base_steps in steps]

    # Initialize recursion
    step_num = 0
    step_bound = max_remaining(steps, step_num)
    path = []
    paths = []
    find_bounded_targeted_goal_paths_rec(steps, goal, step_num, step_bound, path, paths)

    return paths

def find_bounded_targeted_goal_paths_rec(steps, goal, step_num, step_bound, path, paths):
    global rec_calls, failed_paths
    rec_calls += 1

    if step_num == len(steps):
        if sum(path) == goal:
            paths.append(list(path))
        else:
            failed_paths += 1
        return
    elif sum(path) + min_remaining(steps, step_num) > goal:
        return
    elif sum(path) + max_remaining(steps, step_num) < goal:
        return
    else:
        for step in steps[step_num]:
            if step > step_bound:
                continue

            path.append(step)
            find_bounded_targeted_goal_paths_rec(steps, goal, step_num + 1, step, path, paths)
            path.pop()

In [21]:
# Find bounded targeted goal paths (test)
for goal in range(2, 8):
    rec_calls = 0
    failed_paths = 0
    base_steps = [1, 2]
    steps = [base_steps, ]*3
    paths = find_bounded_targeted_goal_paths(steps, goal)
    print_summary(steps, goal, paths)

Goal: 2
Max recursive calls: 15
Num recursive calls: 1
Total paths: 8
Found paths: 0
Failed paths: 0
Failed percentage: 0.0
Skipped paths: 8
Skipped percentage: 100.0
Paths: []

Goal: 3
Max recursive calls: 15
Num recursive calls: 5
Total paths: 8
Found paths: 1
Failed paths: 0
Failed percentage: 0.0
Skipped paths: 7
Skipped percentage: 87.5
Paths: [[1, 1, 1]]

Goal: 4
Max recursive calls: 15
Num recursive calls: 8
Total paths: 8
Found paths: 1
Failed paths: 1
Failed percentage: 12.5
Skipped paths: 6
Skipped percentage: 75.0
Paths: [[2, 1, 1]]

Goal: 5
Max recursive calls: 15
Num recursive calls: 9
Total paths: 8
Found paths: 1
Failed paths: 2
Failed percentage: 25.0
Skipped paths: 5
Skipped percentage: 62.5
Paths: [[2, 2, 1]]

Goal: 6
Max recursive calls: 15
Num recursive calls: 7
Total paths: 8
Found paths: 1
Failed paths: 1
Failed percentage: 12.5
Skipped paths: 6
Skipped percentage: 75.0
Paths: [[2, 2, 2]]

Goal: 7
Max recursive calls: 15
Num recursive calls: 1
Total paths: 8
Found

In [22]:
# Find bounded targeted goal paths (five darts, 300)
goals = [300]
darts = 5
for goal in goals:
    rec_calls = 0
    failed_paths = 0
    base_steps = all_scores.keys()
    steps = [base_steps, ]*5
    paths = find_bounded_targeted_goal_paths(steps, goal)
    print(f"Goal: {goal:1} Num recursive calls: {rec_calls} Max num recursive calls: {max_rec_calls(steps)} Num paths: {len(paths):1} Paths: {paths}\n")
    print_summary(steps, goal, paths)

Goal: 300 Num recursive calls: 216 Max num recursive calls: 150508644 Num paths: 1 Paths: [[60, 60, 60, 60, 60]]

Goal: 300
Max recursive calls: 150508644
Num recursive calls: 216
Total paths: 147008443
Found paths: 1
Failed paths: 42
Failed percentage: 2.8569787654985235e-05
Skipped paths: 147008400
Skipped percentage: 99.99997074997931
Paths: [[60, 60, 60, 60, 60]]



In [23]:
# Find bounded targeted goal paths (six darts, 301)
goals = [301]
darts = 6
for goal in goals:
    rec_calls = 0
    failed_paths = 0
    base_steps = all_scores.keys()
    steps = [base_steps, ]*darts
    paths = find_bounded_targeted_goal_paths(steps, goal)
    print(f"Goal: {goal:1} Num recursive calls: {rec_calls} Max num recursive calls: {max_rec_calls(steps)} Num paths: {len(paths):1} Paths: {paths}\n")
    print_summary(steps, goal, paths)

Goal: 301 Num recursive calls: 175256 Max num recursive calls: 6471871693 Num paths: 277 Paths: [[60, 60, 60, 60, 60, 1], [60, 60, 60, 60, 57, 4], [60, 60, 60, 60, 54, 7], [60, 60, 60, 60, 51, 10], [60, 60, 60, 60, 50, 11], [60, 60, 60, 60, 48, 13], [60, 60, 60, 60, 45, 16], [60, 60, 60, 60, 42, 19], [60, 60, 60, 60, 40, 21], [60, 60, 60, 60, 39, 22], [60, 60, 60, 60, 36, 25], [60, 60, 60, 60, 34, 27], [60, 60, 60, 60, 33, 28], [60, 60, 60, 57, 57, 7], [60, 60, 60, 57, 54, 10], [60, 60, 60, 57, 51, 13], [60, 60, 60, 57, 50, 14], [60, 60, 60, 57, 48, 16], [60, 60, 60, 57, 45, 19], [60, 60, 60, 57, 42, 22], [60, 60, 60, 57, 40, 24], [60, 60, 60, 57, 39, 25], [60, 60, 60, 57, 38, 26], [60, 60, 60, 57, 36, 28], [60, 60, 60, 57, 34, 30], [60, 60, 60, 57, 32, 32], [60, 60, 60, 54, 54, 13], [60, 60, 60, 54, 51, 16], [60, 60, 60, 54, 50, 17], [60, 60, 60, 54, 48, 19], [60, 60, 60, 54, 45, 22], [60, 60, 60, 54, 42, 25], [60, 60, 60, 54, 40, 27], [60, 60, 60, 54, 39, 28], [60, 60, 60, 54, 34, 33

In [24]:
# Find bounded targeted goal paths (nine darts, 501)
goals = [501]
darts = 9
for goal in goals:
    rec_calls = 0
    failed_paths = 0
    base_steps = all_scores.keys()
    steps = [base_steps, ]*darts
    paths = find_bounded_targeted_goal_paths(steps, goal)
    print_summary(steps, goal, paths)

Goal: 501
Max recursive calls: 514559102697244
Num recursive calls: 98595
Total paths: 502592611936843
Found paths: 101
Failed paths: 20636
Failed percentage: 4.1059099377674835e-09
Skipped paths: 502592611916106
Skipped percentage: 99.99999999587399
Paths: [[60, 60, 60, 60, 60, 60, 60, 60, 21], [60, 60, 60, 60, 60, 60, 60, 57, 24], [60, 60, 60, 60, 60, 60, 60, 54, 27], [60, 60, 60, 60, 60, 60, 60, 51, 30], [60, 60, 60, 60, 60, 60, 60, 48, 33], [60, 60, 60, 60, 60, 60, 60, 45, 36], [60, 60, 60, 60, 60, 60, 60, 42, 39], [60, 60, 60, 60, 60, 60, 57, 57, 27], [60, 60, 60, 60, 60, 60, 57, 54, 30], [60, 60, 60, 60, 60, 60, 57, 51, 33], [60, 60, 60, 60, 60, 60, 57, 50, 34], [60, 60, 60, 60, 60, 60, 57, 48, 36], [60, 60, 60, 60, 60, 60, 57, 45, 39], [60, 60, 60, 60, 60, 60, 57, 42, 42], [60, 60, 60, 60, 60, 60, 54, 54, 33], [60, 60, 60, 60, 60, 60, 54, 51, 36], [60, 60, 60, 60, 60, 60, 54, 48, 39], [60, 60, 60, 60, 60, 60, 54, 45, 42], [60, 60, 60, 60, 60, 60, 51, 51, 39], [60, 60, 60, 60, 60

In [25]:
# Find targeted goal paths
def max_remaining(steps, step_num):
    selected_steps = steps[step_num:]
    return sum([max(base_steps) for base_steps in selected_steps])

def min_remaining(steps, step_num):
    selected_steps = steps[step_num:]
    return sum([min(base_steps) for base_steps in selected_steps])

def find_targeted_goal_paths(steps, goal):
    # Preprocess steps
    steps = [sorted(base_steps, reverse=True) for base_steps in steps]

    # Initialize recursion
    step_num = 0
    step_bound = max_remaining(steps, step_num)
    path = []
    paths = []
    find_targeted_goal_paths_rec(steps, goal, step_num, step_bound, path, paths)

    return paths

def find_targeted_goal_paths_rec(steps, goal, step_num, step_bound, path, paths):
    global rec_calls, failed_paths
    rec_calls += 1

    if step_num == len(steps):
        if sum(path) == goal:
            paths.append(list(path))
        else:
            failed_paths += 1
        return
    elif sum(path) + min_remaining(steps, step_num) > goal:
        return
    elif sum(path) + max_remaining(steps, step_num) < goal:
        return
    else:
        for step in steps[step_num]:
            # if step > step_bound:
            #     continue

            path.append(step)
            find_targeted_goal_paths_rec(steps, goal, step_num + 1, step, path, paths)
            path.pop()

In [26]:
# Find targeted goal paths (nine darts, 501)
goals = [501]
darts = 9
for goal in goals:
    rec_calls = 0
    failed_paths = 0
    base_steps = all_scores.keys()
    steps = [base_steps, ]*darts
    # paths = find_targeted_goal_paths(steps, goal)
    # print_summary(steps, goal, paths)

In [27]:
# Find (partly) bounded paths using goal targeting
def find_paths(steps, bounded, goal):
    # Maximal possible sum
    def max_sum(steps, bounded, step_bound=-1):
        ceiling = 0
        for base_steps, base_bounded in zip(steps, bounded):
            max_step = max(base_steps)

            if base_bounded and step_bound != -1:
                max_step = min([max_step, step_bound])
            stats["ifs"] += 1

            step_bound = max_step
            ceiling += max_step

        return ceiling

    # Minimal possible sum
    def min_sum(steps):
        return sum([min(base_steps) for base_steps in steps])

    # Stepping
    def recur(steps, bounded, goal, step_num, step_bound, path, paths, stats):
        stats["calls"] += 1

        # Prepare checking variables
        path_sum = sum(path)
        min_add = min_sum(steps[step_num:])
        max_add = max_sum(steps[step_num:], bounded[step_num:], step_bound)

        if step_num == len(steps):
            # Final step reached
            if path_sum == goal:
                # Goal reached
                paths.append(list(path))
            else:
                # Should not occur
                stats["fails"] += 1
            stats["ifs"] += 2
            return
        elif path_sum + max_add < goal: # or path_sum + min_add > goal:
            # Path sum too small to reach goal
            stats["preundershoots"] += 1
            stats["ifs"] += 2
            return
        elif path_sum + min_add > goal:
            # Path sum too large to reach goal
            stats["preovershoots"] += 1
            stats["ifs"] += 3
            return
        else:
            stats["ifs"] += 3

            # Prepare checking variables
            min_add -= min(steps[step_num])
            max_add = max_sum(steps[step_num + 1:], bounded[step_num + 1:], step_bound)

            # Try stepping
            for step in steps[step_num]:
                stats["fors"] += 1

                stats["ifs"] += 1
                if bounded[step_num] and step > step_bound:
                    # Step larger than bound
                    stats["foroverbound"] += 1
                    continue

                stats["ifs"] += 1
                if path_sum + step + min_add > goal:
                    # Step too large to reach goal
                    stats["forovershoots"] += 1
                    continue

                stats["ifs"] += 1
                if path_sum + step + max_add < goal:
                    # Step too small to reach goal
                    stats["forundershoots"] += 1
                    return

                path.append(step)
                recur(steps, bounded, goal, step_num + 1, step, path, paths, stats)
                path.pop()

    # Order possible steps
    steps = [sorted(base_steps, reverse=True) for base_steps in steps]

    # Initialize recursion
    stats = {
        "calls": 0,
        "ifs": 0,
        "preovershoots": 0,
        "preundershoots": 0,
        "fors": 0,
        "foroverbound": 0,
        "forovershoots": 0,
        "forundershoots": 0,
        "fails": 0
    }
    step_num = 0
    step_bound = max_sum(steps, bounded)
    path = []
    paths = []

    recur(steps, bounded, goal, step_num, step_bound, path, paths, stats)

    return paths, stats

In [28]:
# Find (partly) bounded paths using goal targeting (test)
num_steps = 3
for goal in range(2, 8):
    base_steps = [1, 2]
    steps = [base_steps, ] * num_steps
    bounded = [True] * num_steps
    paths, stats = find_paths(steps, bounded, goal)
    print()
    print_stats_summary(steps, goal, paths, stats)


Goal: 2
Max recursive calls: 15
Num recursive calls: 1
Num ifs: 9
Num pre overshoots: 1
Num pre undershoots: 0
Num for steps: 0
Num for over bound: 0
Num for overshoots: 0
Num for undershoots: 0
Total paths: 8
Found paths: 0
Failed paths: 0
Failed percentage: 0.0
Skipped paths: 8
Skipped percentage: 100.0
Paths: []


Goal: 3
Max recursive calls: 15
Num recursive calls: 4
Num ifs: 36
Num pre overshoots: 0
Num pre undershoots: 0
Num for steps: 6
Num for over bound: 2
Num for overshoots: 1
Num for undershoots: 0
Total paths: 8
Found paths: 1
Failed paths: 0
Failed percentage: 0.0
Skipped paths: 7
Skipped percentage: 87.5
Paths: [[1, 1, 1]]


Goal: 4
Max recursive calls: 15
Num recursive calls: 5
Num ifs: 42
Num pre overshoots: 0
Num pre undershoots: 1
Num for steps: 6
Num for over bound: 1
Num for overshoots: 1
Num for undershoots: 0
Total paths: 8
Found paths: 1
Failed paths: 0
Failed percentage: 0.0
Skipped paths: 7
Skipped percentage: 87.5
Paths: [[2, 1, 1]]


Goal: 5
Max recursive ca

In [29]:
# Find (partly) bounded paths using goal targeting (nine darts, 501)
goals = [501]
darts = 9
for goal in goals:
    base_steps = all_scores.keys()
    steps = [base_steps, ] * darts
    bounded = [True] * darts
    paths, stats = find_paths(steps, bounded, goal)
    print_stats_summary(steps, goal, paths, stats)

Goal: 501
Max recursive calls: 514559102697244
Num recursive calls: 746
Num ifs: 8241
Num pre overshoots: 0
Num pre undershoots: 422
Num for steps: 2064
Num for over bound: 614
Num for overshoots: 482
Num for undershoots: 223
Total paths: 502592611936843
Found paths: 101
Failed paths: 0
Failed percentage: 0.0
Skipped paths: 502592611936742
Skipped percentage: 99.9999999999799
Paths: [[60, 60, 60, 60, 60, 60, 60, 60, 21], [60, 60, 60, 60, 60, 60, 60, 57, 24], [60, 60, 60, 60, 60, 60, 60, 54, 27], [60, 60, 60, 60, 60, 60, 60, 51, 30], [60, 60, 60, 60, 60, 60, 60, 48, 33], [60, 60, 60, 60, 60, 60, 60, 45, 36], [60, 60, 60, 60, 60, 60, 60, 42, 39], [60, 60, 60, 60, 60, 60, 57, 57, 27], [60, 60, 60, 60, 60, 60, 57, 54, 30], [60, 60, 60, 60, 60, 60, 57, 51, 33], [60, 60, 60, 60, 60, 60, 57, 50, 34], [60, 60, 60, 60, 60, 60, 57, 48, 36], [60, 60, 60, 60, 60, 60, 57, 45, 39], [60, 60, 60, 60, 60, 60, 57, 42, 42], [60, 60, 60, 60, 60, 60, 54, 54, 33], [60, 60, 60, 60, 60, 60, 54, 51, 36], [60, 

In [31]:
def print_stats_summary(steps, goal, paths, stats):
    num_base_steps = [len(base_steps) for base_steps in steps]
    total_paths = multiply(num_base_steps)
    found_paths = len(paths)
    failed_paths = stats["fails"]
    skipped_paths = total_paths - found_paths - failed_paths

    print(
        f"Goal: {goal}\n"
        f"Max recursive calls: {max_rec_calls(steps)}\n"
        f"Num recursive calls: {stats['calls']}\n"
        f"Num ifs: {stats['ifs']}\n"
        f"Num pre overshoots: {stats['preovershoots']}\n"
        f"Num pre undershoots: {stats['preundershoots']}\n"
        f"Num for steps: {stats['fors']}\n"
        f"Num for over bound: {stats['foroverbound']}\n"
        f"Num for overshoots: {stats['forovershoots']}\n"
        f"Num for undershoots: {stats['forundershoots']}\n"
        f"Total paths: {total_paths}\n"
        f"Found paths: {found_paths}\n"
        f"Failed paths: {failed_paths}\n"
        f"Failed percentage: {failed_paths / total_paths * 100}\n"
        f"Skipped paths: {skipped_paths}\n"
        f"Skipped percentage: {skipped_paths / total_paths * 100}\n"
        # f"Paths: {paths}\n"
    )

def multiply(nums):
    if len(nums) == 0:
        return 0
    else:
        prod = 1
        for num in nums:
            prod *= num

        return prod

def max_rec_calls(steps):
    num_base_steps = [len(base_steps) for base_steps in steps]

    rec_calls = 1

    for end in range(1, len(num_base_steps) + 1):
        rec_calls += multiply(num_base_steps[:end])

    return rec_calls

In [33]:
# Imports
from importlib import reload
import paths, path
reload(paths)
reload(path)
from paths import Paths

# Find (partly) bounded paths using goal targeting (nine darts, 501)
goals = [501]
darts = 9
for goal in goals:
    steps = [all_scores.keys(), ] * (darts - 1)
    steps.append(double_scores.keys())
    bounded = [False]
    bounded.extend([True] * (darts - 2))
    bounded.append(False)
    paths = Paths(steps, bounded, goal)
    print_stats_summary(steps, goal, paths, paths.stats)
    paths.print()

Goal: 501
Max recursive calls: 257418696590022
Num recursive calls: 394
Num ifs: 4682
Num pre overshoots: 0
Num pre undershoots: 152
Num for steps: 987
Num for over bound: 130
Num for overshoots: 244
Num for undershoots: 220
Total paths: 245452205829621
Found paths: 22
Failed paths: 0
Failed percentage: 0.0
Skipped paths: 245452205829599
Skipped percentage: 99.99999999999105

Paths:
#1: [60, 60, 60, 60, 60, 60, 60, 57, 24]: 8! / 7! * 1! = 8
#2: [60, 60, 60, 60, 60, 60, 60, 51, 30]: 8! / 7! * 1! = 8
#3: [60, 60, 60, 60, 60, 60, 60, 45, 36]: 8! / 7! * 1! = 8
#4: [60, 60, 60, 60, 60, 60, 57, 54, 30]: 8! / 6! * 1! = 56
#5: [60, 60, 60, 60, 60, 60, 57, 50, 34]: 8! / 6! * 1! = 56
#6: [60, 60, 60, 60, 60, 60, 57, 48, 36]: 8! / 6! * 1! = 56
#7: [60, 60, 60, 60, 60, 60, 57, 34, 50]: 8! / 6! * 1! = 56
#8: [60, 60, 60, 60, 60, 60, 54, 51, 36]: 8! / 6! * 1! = 56
#9: [60, 60, 60, 60, 60, 60, 51, 50, 40]: 8! / 6! * 1! = 56
#10: [60, 60, 60, 60, 60, 60, 51, 40, 50]: 8! / 6! * 1! = 56
#11: [60, 60, 60